In [0]:
%pip install fastf1

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
# 1. Create the Widgets with the full driver list
dbutils.widgets.text("year", "AUTO", "1. Year (e.g. 2024 or AUTO)")
dbutils.widgets.text("round", "AUTO", "2. Round (e.g. Monaco or AUTO)")

# The complete list with F1 driver codes
all_drivers = [
    "VER", "HAM", "LEC", "PER", "SAI", "NOR", "PIA", "RUS", 
    "ALO", "STR", "GAS", "OCO", "ALB", "TSU", "BOT", "ZHO", 
    "MAG", "HUL", "RIC", "VET", "RAI", "GIO", "MSC", "MAZ", "LAT",
    "ANT", "COL", "BEA", "DOO", "BOR", "LAW", "HAD", "LIN"
]

# Add "ALL" at the beginning of the list for the dropdown menu
widget_options = ["ALL"] + all_drivers

# Default value set to "ALL"
dbutils.widgets.multiselect("drivers", "ALL", widget_options, "3. Select Drivers")

# 2. Read the values selected by the user
input_year = dbutils.widgets.get("year")
input_round = dbutils.widgets.get("round")
selected_drivers_list = dbutils.widgets.get("drivers").split(",")

print(f"Selected drivers for analysis: {selected_drivers_list}")

Selected drivers for analysis: ['ALL']


In [0]:
import fastf1
import pandas as pd
from pyspark.sql.functions import col

# --- READ WIDGETS DIRECTLY HERE ---
input_year = dbutils.widgets.get("year")
input_round = dbutils.widgets.get("round")
raw_drivers_selection = dbutils.widgets.get("drivers").split(",")

# --- 1. AUTOMATIC DETECTION OR MANUAL SELCTION ---
if input_year == "AUTO" or input_round == "AUTO":
    today = pd.Timestamp.now().tz_localize(None)
    schedule = fastf1.get_event_schedule(today.year)
    schedule['EventDate'] = pd.to_datetime(schedule['EventDate']).dt.tz_localize(None)
    completed_races = schedule[schedule['EventDate'] < today]
    latest_race = completed_races.iloc[-1]
    
    selected_year = today.year
    selected_round = latest_race['RoundNumber']
    print(f"AUTO MODE: Detected race {latest_race['EventName']} ({selected_year})")
else:
    selected_year = int(input_year)
    try:
        selected_round = int(input_round)
    except:
        selected_round = input_round

# --- 2. DATA INGESTION (BRONZE) ---
fastf1.Cache.enable_cache('/tmp/')
session = fastf1.get_session(selected_year, selected_round, 'R')
session.load(telemetry=False)

# Get the laps and weather data
laps_df = session.laps
weather_df = session.weather_data

# Data quality check: drop rows with missing Time to avoid errors
laps_df = laps_df.dropna(subset=['Time'])
weather_df = weather_df.dropna(subset=['Time'])

# --- 3. SERIES MERGE (Time-series Merge) ---
combined_df = pd.merge_asof(laps_df.sort_values('Time'), 
                           weather_df.sort_values('Time'), 
                           on='Time', 
                           direction='backward')

# --- 4. PREPARATION FOR SPARK ---
for column in ['LapTime', 'Sector1Time', 'Sector2Time', 'Sector3Time', 'Time']:
    if column in combined_df.columns:
        combined_df[column] = combined_df[column].astype(str)

# CLEAN TABLE NAME: Remove spaces (e.g., "Abu Dhabi" -> "abudhabi")
clean_round = str(selected_round).lower().replace(' ', '')
raw_table_name = f"workspace.default.raw_laps_weather_{clean_round}_{selected_year}"

# Save to RAW table 
spark_combined_df = spark.createDataFrame(combined_df)
spark_combined_df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable(raw_table_name)

# --- 5. CLEANING & FILTERING (SILVER) ---
# Start by filtering out invalid (NaT) LapTimes
df_final = spark.table(raw_table_name).filter(col("LapTime") != 'NaT')

# If "ALL" is not selected, then filter by drivers
if "ALL" not in raw_drivers_selection:
    df_final = df_final.filter(col("Driver").isin(raw_drivers_selection))

# Save
df_final.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("workspace.default.silver_race_comparison")

print(f"SUCCESS! Data has been downloaded and the table now includes the weather.")

core           INFO 	Loading data for Abu Dhabi Grand Prix - Race [v3.8.3]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No 

SUCCESS! Data has been downloaded and the table now includes the weather.


In [0]:
%sql
SELECT 
    Driver,
    LapNumber AS Lap,
    
    -- OUR FILTER: If the position is 0 (or invalid), set it to NULL (invisible)
    CASE 
        WHEN CAST(Position AS INT) = 0 OR Position IS NULL THEN NULL 
        ELSE CAST(Position AS INT) 
    END AS Race_Position,
    
    Compound AS Tyre_Type,
    CAST(TyreLife AS INT) AS Tyre_Age_Laps,
    ROUND((TRY_CAST(SUBSTRING(LapTime, 11, 2) AS INT) * 60) + TRY_CAST(SUBSTRING(LapTime, 14, 6) AS FLOAT), 3) AS Lap_Time_Seconds,
    CAST(TrackTemp AS FLOAT) AS Track_Temperature,
    CAST(AirTemp AS FLOAT) AS Air_Temperature

FROM workspace.default.silver_race_comparison
WHERE LapTime != 'NaT'
ORDER BY LapNumber ASC;

Driver,Lap,Race_Position,Tyre_Type,Tyre_Age_Laps,Lap_Time_Seconds,Track_Temperature,Air_Temperature
LEC,1.0,6,SOFT,4,96.143,28.4,24.6
STR,1.0,14,MEDIUM,1,100.334,28.4,24.6
RAI,1.0,17,MEDIUM,1,101.61,28.4,24.6
VET,1.0,15,MEDIUM,1,100.689,28.4,24.6
NOR,1.0,5,SOFT,4,95.658,28.4,24.6
ALO,1.0,11,HARD,2,98.679,28.4,24.6
HAM,1.0,1,MEDIUM,4,91.686,28.4,24.6
SAI,1.0,4,SOFT,4,94.819,28.4,24.6
GIO,1.0,13,MEDIUM,1,99.822,28.4,24.6
RUS,1.0,19,MEDIUM,1,102.644,28.4,24.6


Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

Databricks visualization. Run in Databricks to view.

In [0]:
%sql
SELECT 
    Driver,
    LapNumber AS Lap,
    
    -- Insert new tire data --
    Compound AS Tyre_Type,
    CAST(TyreLife AS INT) AS Tyre_Age_Laps,

    -- Lap time in seconds --
    ROUND((TRY_CAST(SUBSTRING(LapTime, 11, 2) AS INT) * 60) + TRY_CAST(SUBSTRING(LapTime, 14, 6) AS FLOAT), 3) AS Lap_Time_Seconds
FROM workspace.default.silver_race_comparison
WHERE LapTime != 'NaT'
ORDER BY LapNumber ASC;

Driver,Lap,Tyre_Type,Tyre_Age_Laps,Lap_Time_Seconds
LEC,1.0,SOFT,4,96.143
STR,1.0,MEDIUM,1,100.334
RAI,1.0,MEDIUM,1,101.61
VET,1.0,MEDIUM,1,100.689
NOR,1.0,SOFT,4,95.658
ALO,1.0,HARD,2,98.679
HAM,1.0,MEDIUM,4,91.686
SAI,1.0,SOFT,4,94.819
GIO,1.0,MEDIUM,1,99.822
RUS,1.0,MEDIUM,1,102.644


Databricks visualization. Run in Databricks to view.